# AndinaLog 03B | Flota | Tratamiento v2

Contrato didáctico: datos originales visibles, decisiones trazables y tres salidas CSV.


In [ ]:
from pathlib import Path
import hashlib
import sys
import pandas as pd
import numpy as np

ENTORNO = "auto"  # auto, local, drive
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
VERSION_DIAGNOSTICO_REQUERIDA = "GIAD-M3-S4-FLOTA-diagnostico-didactico-v2"
VERSION_TRATAMIENTO = "GIAD-M3-S4-FLOTA-tratamiento-didactico-v2"
COLUMNAS_BRONZE = ["camion_id", "centro_distribucion_base", "capacidad_kg", "tipo_camion"]
CENTROS_PERMITIDOS = {"Cochabamba", "La Paz", "Santa Cruz", "Oruro", "Tarija"}
TIPOS_PERMITIDOS = {"Seco", "Refrigerado"}
UMBRAL_REVISION_CAPACIDAD_KG = 750

def encontrar_raiz():
    if ENTORNO == "drive" or (ENTORNO == "auto" and "google.colab" in sys.modules):
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(RUTA_PROYECTO_DRIVE)
        if not (raiz / "proyecto-integrador/01_diagnostico/andinalog_flota/salidas/andinalog_flota_didactico_v2_diagnosticado.csv").is_file():
            raise FileNotFoundError(raiz)
        return raiz
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "proyecto-integrador/01_diagnostico/andinalog_flota/salidas/andinalog_flota_didactico_v2_diagnosticado.csv").is_file():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz de practicasNotebookColab")

RAIZ = encontrar_raiz()
RUTA_BRONZE = RAIZ / "datasets/AndinaLog_03B_Bronce/andinalog_flota.csv"
ENTRADA = RAIZ / "proyecto-integrador/01_diagnostico/andinalog_flota/salidas/andinalog_flota_didactico_v2_diagnosticado.csv"
SALIDAS = RAIZ / "proyecto-integrador/02_tratamiento/andinalog_flota/salidas"
df = pd.read_csv(ENTRADA, dtype="string", encoding="utf-8-sig", keep_default_na=False)
bronze = pd.read_csv(RUTA_BRONZE, dtype="string", encoding="utf-8-sig", keep_default_na=False)
requeridas=["fila_bronze","en_cuarentena",*COLUMNAS_BRONZE]
requeridas += [f"{c}_{s}" for c in COLUMNAS_BRONZE for s in ("en_cuarentena","motivo")]
faltantes = sorted(set(requeridas) - set(df.columns))
if faltantes: raise ValueError(f"Faltan columnas del diagnóstico: {faltantes}")
if list(bronze.columns) != COLUMNAS_BRONZE: raise ValueError("Esquema Bronze inesperado")
if len(df) != len(bronze): raise ValueError("El diagnóstico no contiene todas las filas Bronze")
if df["fila_bronze"].duplicated().any(): raise ValueError("fila_bronze debe ser único")
if not df["fila_bronze"].eq(pd.Series(range(1, len(df)+1), dtype="string")).all():
    raise ValueError("Orden o numeración Bronze inesperados")
pd.testing.assert_frame_equal(df[COLUMNAS_BRONZE], bronze)
huella = hashlib.sha256(RUTA_BRONZE.read_bytes()).hexdigest()
if not df["en_cuarentena"].isin(["True", "False"]).all():
    raise ValueError("en_cuarentena debe ser True o False")
original = df.copy(deep=True)
df=df.rename(columns={"en_cuarentena":"en_cuarentena_diagnostico"})
print("Filas de entrada:", len(df))


## 1. Valores tratados y acciones

Las columnas `*_tratado` son nuevas. El original no se sobrescribe. `acciones_tratamiento` y `motivos_tratamiento` permiten auditar las decisiones. El indicador `camion_id_normalizado` señala expresamente el cambio de ID.


In [ ]:
df["acciones_tratamiento"] = ""
df["motivos_tratamiento"] = ""
def anotar(mascara, accion, motivo):
    mascara = pd.Series(mascara, index=df.index).fillna(False).astype(bool)
    for columna, texto in [("acciones_tratamiento", accion), ("motivos_tratamiento", motivo)]:
        previo = df.loc[mascara, columna]
        df.loc[mascara, columna] = previo.where(previo.eq(""), previo + " | ") + texto

df["camion_id_tratado"] = df["camion_id"].str.strip().str.upper()
df["centro_distribucion_base_tratado"] = df["centro_distribucion_base"]
df["tipo_camion_tratado"] = df["tipo_camion"]
df["capacidad_kg_tratada"] = pd.to_numeric(df["capacidad_kg"].str.strip(), errors="coerce")
df["camion_id_normalizado"] = df["camion_id_tratado"].ne(df["camion_id"])

# Un ID normalizado se acepta solo si corresponde al patrón y no crea
# una variante contradictoria del mismo camión.
patron_id = df["camion_id_tratado"].str.fullmatch(r"CAM-\d{2}").fillna(False)
firma_atributos = pd.util.hash_pandas_object(df[["centro_distribucion_base_tratado",
    "capacidad_kg_tratada", "tipo_camion_tratado"]], index=False)
variantes = firma_atributos.groupby(df["camion_id_tratado"], dropna=False).transform("nunique")
conflicto = df["camion_id_tratado"].ne("") & variantes.gt(1)
normalizacion_aceptada = df["camion_id_normalizado"] & patron_id & ~conflicto
anotar(normalizacion_aceptada, "NORMALIZAR_CAMION_ID", "ID normalizado sin conflicto de atributos")

# Conservar la primera aparición, identificada por fila_bronze.
copia = df.duplicated(["camion_id_tratado", "centro_distribucion_base_tratado",
    "capacidad_kg_tratada", "tipo_camion_tratado"], keep="first") & ~conflicto
anotar(copia, "EXCLUIR_COPIA", "Copia exacta posterior del mismo camión")
anotar(conflicto, "CUARENTENA_CONFLICTO", "Mismo ID normalizado con atributos contradictorios")


## 2. Validación final y destino

Los datos no recuperables quedan en cuarentena. Una capacidad positiva menor que 750 kg es una advertencia para revisar la ficha técnica y puede permanecer en Silver; no constituye un mínimo universal. No se imputa ningún campo de este maestro.


In [ ]:
df["motivo_cuarentena_final"] = ""
def cuarentena_si(mascara, motivo):
    mascara = pd.Series(mascara, index=df.index).fillna(True).astype(bool)
    previo = df.loc[mascara, "motivo_cuarentena_final"]
    df.loc[mascara, "motivo_cuarentena_final"] = previo.where(previo.eq(""), previo + " | ") + motivo

cuarentena_si(~patron_id, "ID de camión inválido")
cuarentena_si(conflicto, "ID con atributos contradictorios")
cuarentena_si(copia, "Copia exacta excluida del Silver")
cuarentena_si(~df["centro_distribucion_base_tratado"].isin(CENTROS_PERMITIDOS), "Centro inválido o faltante")
cuarentena_si(~df["tipo_camion_tratado"].isin(TIPOS_PERMITIDOS), "Tipo inválido o faltante")
cuarentena_si(df["capacidad_kg_tratada"].isna() | df["capacidad_kg_tratada"].le(0),
             "Capacidad no numérica, faltante o no positiva")
df["capacidad_revisar_ficha"] = df["capacidad_kg_tratada"].gt(0) & df["capacidad_kg_tratada"].lt(UMBRAL_REVISION_CAPACIDAD_KG)
anotar(df["capacidad_revisar_ficha"], "REVISAR_FICHA_CAPACIDAD", "Capacidad inferior a 750 kg; verificar ficha técnica")

df["en_cuarentena_final"] = df["motivo_cuarentena_final"].ne("")
df["decision_tratamiento"] = np.where(df["en_cuarentena_final"], "CUARENTENA", "SILVER")
silver = df.loc[~df["en_cuarentena_final"]].copy()
cuarentena_final = df.loc[df["en_cuarentena_final"]].copy()


## 3. Comprobaciones y exportación

Se exportan exactamente tres CSV: Silver, cuarentena final y reporte de calidad. Ambos conservan las columnas originales y de diagnóstico para mantener trazabilidad. Una fila diagnosticada como crítica puede quedar en Silver tras una corrección inequívoca; `en_cuarentena_diagnostico` y `en_cuarentena_final` registran ambas etapas.


In [ ]:
pd.testing.assert_frame_equal(df[[c for c in original.columns if c!="en_cuarentena"]], original[[c for c in original.columns if c!="en_cuarentena"]])
assert len(df) == len(silver) + len(cuarentena_final)
assert silver["camion_id_tratado"].is_unique
assert silver["motivo_cuarentena_final"].eq("").all()
assert cuarentena_final["motivo_cuarentena_final"].ne("").all()
assert silver["camion_id_tratado"].str.fullmatch(r"CAM-\d{2}").fillna(False).all()
assert silver["centro_distribucion_base_tratado"].isin(CENTROS_PERMITIDOS).all()
assert silver["tipo_camion_tratado"].isin(TIPOS_PERMITIDOS).all()
assert silver["capacidad_kg_tratada"].gt(0).all()
assert hashlib.sha256(RUTA_BRONZE.read_bytes()).hexdigest() == huella

SALIDAS.mkdir(parents=True, exist_ok=True)
ruta_silver = SALIDAS / "andinalog_flota_didactico_v2_silver.csv"
ruta_cuarentena = SALIDAS / "andinalog_flota_didactico_v2_cuarentena_final.csv"
silver.to_csv(ruta_silver, index=False, encoding="utf-8-sig")
cuarentena_final.to_csv(ruta_cuarentena, index=False, encoding="utf-8-sig")
print("Entrada:", len(df), "| Silver:", len(silver), "| Cuarentena final:", len(cuarentena_final))
print("IDs normalizados:", int(normalizacion_aceptada.sum()), "| Copias excluidas:", int(copia.sum()))
print("Silver:", ruta_silver)
print("Cuarentena:", ruta_cuarentena)
display(df[["fila_bronze", "camion_id", "camion_id_tratado", "en_cuarentena_diagnostico",
    "decision_tratamiento", "acciones_tratamiento", "motivo_cuarentena_final"]].tail(15))

# Third treatment output and published distinction between prior and final quarantine.
assert "en_cuarentena_diagnostico" in df and "en_cuarentena" not in df
assert len(silver)+len(cuarentena_final)==len(original)
metricas={"filas_entrada":len(df),"filas_silver":len(silver),
          "filas_cuarentena_final":len(cuarentena_final),
          "filas_recuperadas":int((df["en_cuarentena_diagnostico"].eq("True") & ~df["en_cuarentena_final"]).sum())}
for campo in ["categoria_imputada","temperatura_imputada","vencimiento_imputado", "capacidad_revisar_ficha"]:
    if campo in df: metricas[campo]=int(df[campo].fillna(False).astype(bool).sum())
reporte_calidad=pd.DataFrame([{"metrica":k,"valor":v} for k,v in metricas.items()])
reporte_calidad.to_csv(SALIDAS/("andinalog_flota_didactico_v2_reporte_calidad.csv"),index=False,encoding="utf-8-sig")
print(metricas)
